## **Import Required libraries**

In [0]:
from pyspark.sql.functions import current_date,col,year, month
from delta.tables import DeltaTable
from datetime import datetime

## **Load From Date**

In [0]:
today = datetime.today()
date_str = today.strftime("%Y-%m-%d")


In [0]:
# Get values from previous task
month_list = dbutils.jobs.taskValues.get(taskKey="silver", key="month_list")
year_list = dbutils.jobs.taskValues.get(taskKey="silver", key="year_list")


In [0]:
#month_list = [1,2]
#year_list = [2024]

## **Path - Source**

In [0]:
cast = '/Volumes/movie_project/silver/cast'
crew = '/Volumes/movie_project/silver/crew'
genre = '/Volumes/movie_project/silver/genre'
movie = '/Volumes/movie_project/silver/movie'
prod_company = '/Volumes/movie_project/silver/prod_company'
prod_country = '/Volumes/movie_project/silver/prod_country'
spoken_language = '/Volumes/movie_project/silver/spoken_language'
country = '/Volumes/movie_project/silver/country'

## **Path - Sink**

In [0]:
cast_dim = '/Volumes/movie_project/gold/cast_dim'
crew_dim = '/Volumes/movie_project/gold/crew_dim'
genre_dim = '/Volumes/movie_project/gold/genre_dim'
movie_dim = '/Volumes/movie_project/gold/movie_dim'
prod_company_dim = '/Volumes/movie_project/gold/prod_company_dim'
prod_country_dim = '/Volumes/movie_project/gold/prod_country_dim'
language_dim = '/Volumes/movie_project/gold/language_dim'
country_dim = '/Volumes/movie_project/gold/country_dim'
cast_bridge = '/Volumes/movie_project/gold/cast_bridge'
crew_bridge = '/Volumes/movie_project/gold/crew_bridge'
genre_bridge = '/Volumes/movie_project/gold/genre_bridge'
country_bridge = '/Volumes/movie_project/gold/country_bridge'
movie_fact = '/Volumes/movie_project/gold/movie_fact'
prod_company_bridge = '/Volumes/movie_project/gold/prod_company_bridge'
prod_country_bridge = '/Volumes/movie_project/gold/prod_country_bridge'
language_bridge = '/Volumes/movie_project/gold/language_bridge'

## **Country - Dim**

In [0]:
df_country = spark.read.format('delta').load(country)
#persist in memory

In [0]:
#Transfrom

df_country_dim = df_country.select(col('origin_country').alias('country'))\
                    .dropDuplicates(['country'])\
                    .withColumn('modified_date_gold',current_date())

In [0]:
#Load
df_country_dim.write.format('delta') \
        .mode('overwrite') \
        .clusterBy('country')\
        .option('path',country_dim)\
        .save()

## **Country - Bridge**

In [0]:
#Transform

df_country_bridge = df_country\
                .filter(f'modified_date_silver >= to_date({date_str})')\
                .select('movie_id',col('origin_country').alias('country'),'release_year','release_month')\
                .withColumn('modified_date_gold',current_date())

In [0]:
# Load

if DeltaTable.isDeltaTable(spark,country_bridge):

    delta_table = DeltaTable.forPath(spark, country_bridge)

    # MERGE (UPSERT)
    delta_table.alias('t').merge(
        df_country_bridge.alias('s'),
        ((col('s.movie_id') == col('t.movie_id')) & (col('s.country') == col('t.country')) & col('t.release_month').isin(month_list) & col('t.release_year').isin(year_list))
    ).whenMatchedUpdateAll()\
        .whenNotMatchedInsertAll()\
        .execute()

else:
    # Initial Load
    df_country_bridge.write.format('delta') \
        .mode('overwrite') \
        .partitionBy('release_year','release_month')\
        .option('path',country_bridge)\
        .save()



## **Cast - Dim**

In [0]:
#Read

df_cast = spark.read.format('delta').load(cast)

#need to persist in the memory

In [0]:
#Transform

df_cast_dim = df_cast.select('id','name','gender','popularity')\
                .dropDuplicates(['id'])\
                .withColumn('modified_date_gold',current_date())

In [0]:
# Load

df_cast_dim.write.format('delta') \
        .mode('overwrite') \
        .clusterBy('id')\
        .option('path',cast_dim)\
        .save()

## **Cast - Bridge**

In [0]:
#Transform

df_cast_bridge = df_cast.filter(f'modified_date_silver >= to_date({date_str})')\
                    .select('id','movie_id','credit_id',col('known_for_department').alias('department'),'order','release_year','release_month')\
                    .withColumn('modified_date_gold',current_date())


In [0]:
# Load

if DeltaTable.isDeltaTable(spark,cast_bridge):

    delta_table = DeltaTable.forPath(spark, cast_bridge)

    # MERGE (UPSERT)
    delta_table.alias('t').merge(
        df_cast_bridge.alias('s'),
        ((col('s.movie_id') == col('t.movie_id')) & (col('s.credit_id') == col('t.credit_id')) & col('t.release_month').isin(month_list) & col('t.release_year').isin(year_list))
    ).whenMatchedUpdateAll()\
        .whenNotMatchedInsertAll()\
        .execute()

else:
    # Initial Load
    df_cast_bridge.write.format('delta') \
        .mode('overwrite') \
        .partitionBy('release_year','release_month')\
        .option('path',cast_bridge)\
        .save()



## **Crew - dim**

In [0]:
#Read

df_crew = spark.read.format('delta').load(crew)

#need to persist in the memory

In [0]:
#Transformation

df_crew_dim = df_crew.select('id','name','gender','popularity')\
                .dropDuplicates(['id'])\
                .withColumn('modified_date_gold',current_date())

In [0]:
#Load
df_crew_dim.write.format('delta') \
        .mode('overwrite') \
        .clusterBy('id')\
        .option('path',crew_dim)\
        .save()

## **Crew - Bridge**

In [0]:
#Transform 

df_crew_bridge = df_crew.filter(f'modified_date_silver >= to_date({date_str})')\
                .select('id','movie_id','credit_id','job','department','release_year','release_month')\
                .withColumn('modified_date_gold',current_date())

In [0]:
df_crew_bridge.select('release_year').distinct().display()

release_year
2024
2023


In [0]:
# Load

if DeltaTable.isDeltaTable(spark,crew_bridge):

    delta_table = DeltaTable.forPath(spark, crew_bridge)

    # MERGE (UPSERT)
    delta_table.alias('t').merge(
        df_crew_bridge.alias('s'),
        ((col('s.movie_id') == col('t.movie_id')) & (col('s.credit_id') == col('t.credit_id')) & col('t.release_month').isin(month_list) & col('t.release_year').isin(year_list))
    ).whenMatchedUpdateAll()\
        .whenNotMatchedInsertAll()\
        .execute()

else:
    # Initial Load
    df_crew_bridge.write.format('delta') \
        .mode('overwrite') \
        .partitionBy('release_year','release_month')\
        .option('path',crew_bridge)\
        .save()



## **Genre - Dim**

In [0]:
#Read

df_genre = spark.read.format('delta').load(genre)

#need to persist in the memory

In [0]:
#Transformation

df_genre_dim = df_genre.select('id','name')\
                    .dropDuplicates(['id'])\
                    .withColumn('modified_date_gold',current_date())


In [0]:
#Load
df_genre_dim.write.format('delta') \
        .mode('overwrite') \
        .clusterBy('id')\
        .option('path',genre_dim)\
        .save()

## **Genre - Bridge**

In [0]:
#Transfrom

df_genre_bridge = df_genre.filter(f'modified_date_silver >= to_date({date_str})')\
                    .select('id','movie_id','release_year','release_month')\
                    .withColumn('modified_date_gold',current_date())


In [0]:
# Load

if DeltaTable.isDeltaTable(spark,genre_bridge):

    delta_table = DeltaTable.forPath(spark, genre_bridge)

    # MERGE (UPSERT)
    delta_table.alias('t').merge(
        df_genre_bridge.alias('s'),
        ((col('s.movie_id') == col('t.movie_id')) & (col('s.id') == col('t.id')) & col('t.release_month').isin(month_list) & col('t.release_year').isin(year_list))
    ).whenMatchedUpdateAll()\
        .whenNotMatchedInsertAll()\
        .execute()

else:
    # Initial Load
    df_genre_bridge.write.format('delta') \
        .mode('overwrite') \
        .partitionBy('release_year','release_month')\
        .option('path',genre_bridge)\
        .save()



## **Movie - Dim**

In [0]:
#Read

df_movie = spark.read.format('delta').load(movie).filter(f'modified_date_silver >= to_date({date_str})')

#need to persist in the memory

In [0]:
#Transformation

df_movie_dim = df_movie.select('id',col('title').alias('name'),'original_language',
                               'overview','status','tagline','is_adult','release_year','release_month')\
                        .dropDuplicates(['id'])\
                        .withColumn('modified_date_gold',current_date())


In [0]:
# Load

if DeltaTable.isDeltaTable(spark,movie_dim):

    delta_table = DeltaTable.forPath(spark, movie_dim)

    # MERGE (UPSERT)
    delta_table.alias('t').merge(
        df_movie_dim.alias('s'),
        ((col('s.id') == col('t.id')) & col('t.release_month').isin(month_list) & col('t.release_year').isin(year_list))
    ).whenMatchedUpdateAll()\
        .whenNotMatchedInsertAll()\
        .execute()

else:
    # Initial Load
    df_movie_dim.write.format('delta') \
        .mode('overwrite') \
        .partitionBy('release_year','release_month')\
        .option('path',movie_dim)\
        .save()



## **Movie - Fact**

In [0]:
#Transfrom

df_movie_fact = df_movie\
                .select('id','imdb_id','vote_average',
                    'vote_count','revenue','budget','runtime', 'popularity','release_date','release_year','release_month')\
                    .withColumn('created_date_gold',current_date())

In [0]:
# Load

if DeltaTable.isDeltaTable(spark,movie_fact):

    delta_table = DeltaTable.forPath(spark, movie_fact)

    # MERGE (UPSERT)
    delta_table.alias('t').merge(
        df_movie_fact.alias('s'),
        ((col('s.id') == col('t.id')) & col('t.release_month').isin(month_list) & col('t.release_year').isin(year_list))
    ).whenMatchedUpdateAll()\
        .whenNotMatchedInsertAll()\
        .execute()

else:
    # Initial Load
    df_movie_fact.write.format('delta') \
        .mode('overwrite') \
        .partitionBy('release_year','release_month')\
        .option('path',movie_fact)\
        .save()



## **Production Company - Dim**

In [0]:
#Read

df_prod_company = spark.read.format('delta').load(prod_company)


#need to persist in the memory

In [0]:
#Transformation

df_prod_company_dim = df_prod_company.select('id','name','origin_country')\
                        .dropDuplicates(['id'])\
                        .withColumn('modified_date_gold',current_date())


In [0]:
#Load
df_prod_company_dim.write.format('delta') \
        .mode('overwrite') \
        .clusterBy('id')\
        .option('path',prod_company_dim)\
        .save()



## **Production Company - Bridge**

In [0]:
#Transfrom

df_prod_company_bridge = df_prod_company\
                    .filter(f'modified_date_silver >= to_date({date_str})')\
                    .select('id','movie_id','release_year','release_month')\
                    .withColumn('created_date_gold',current_date())

In [0]:
# Load

if DeltaTable.isDeltaTable(spark,prod_company_bridge):

    delta_table = DeltaTable.forPath(spark, prod_company_bridge)

    # MERGE (UPSERT)
    delta_table.alias('t').merge(
        df_prod_company_bridge.alias('s'),
        ((col('s.movie_id') == col('t.movie_id'))&(col('s.id') == col('t.id')) & col('t.release_month').isin(month_list) & col('t.release_year').isin(year_list))
    ).whenMatchedUpdateAll()\
        .whenNotMatchedInsertAll()\
        .execute()

else:
    # Initial Load
    df_prod_company_bridge.write.format('delta') \
        .mode('overwrite') \
        .partitionBy('release_year','release_month')\
        .option('path',prod_company_bridge)\
        .save()



## **Production Country - Dim**

In [0]:
#Read

df_prod_country = spark.read.format('delta').load(prod_country)

#need to persist in the memory

In [0]:
#Transfrom

df_prod_country_dim = df_prod_country.select('id','name')\
                        .dropDuplicates(['id'])\
                        .withColumn('modified_date_gold',current_date())



In [0]:
#Load

df_prod_country_dim.write.format('delta') \
        .mode('overwrite') \
        .clusterBy('id')\
        .option('path',prod_country_dim)\
        .save()



## **Production Country - Bridge**

In [0]:
#Transformation

df_prod_country_bridge = df_prod_country.filter(f'modified_date_silver >= to_date({date_str})')\
                    .select('id','movie_id','release_year','release_month')\
                    .withColumn('created_date_gold',current_date())



In [0]:
# Load

if DeltaTable.isDeltaTable(spark,prod_country_bridge):

    delta_table = DeltaTable.forPath(spark, prod_country_bridge)

    # MERGE (UPSERT)
    delta_table.alias('t').merge(
        df_prod_country_bridge.alias('s'),
        ((col('s.movie_id') == col('t.movie_id'))&(col('s.id') == col('t.id')) & col('t.release_month').isin(month_list) & col('t.release_year').isin(year_list))
    ).whenMatchedUpdateAll()\
        .whenNotMatchedInsertAll()\
        .execute()

else:
    # Initial Load
    df_prod_country_bridge.write.format('delta') \
        .mode('overwrite') \
        .partitionBy('release_year','release_month')\
        .option('path',prod_country_bridge)\
        .save()



## **Languages - Dim**

In [0]:
#Read

df_language = spark.read.format('delta').load(spoken_language)
#need to persist in the memory

In [0]:
#Transformation

df_language_dim = df_language.select('id',col('english_name').alias('name'))\
                    .dropDuplicates(['id'])\
                    .withColumn('modified_date_gold',current_date())


In [0]:
#Load
df_language_dim.write.format('delta') \
        .mode('overwrite') \
        .clusterBy('id')\
        .option('path',language_dim)\
        .save()



## **Languages - Bridge**

In [0]:
#Transform

df_language_bridge = df_language\
                    .filter(f'modified_date_silver >= to_date({date_str})')\
                    .select('id','movie_id','release_year','release_month')\
                    .withColumn('created_date_gold',current_date())


In [0]:
# Load

if DeltaTable.isDeltaTable(spark,language_bridge):

    delta_table = DeltaTable.forPath(spark, language_bridge)

    # MERGE (UPSERT)
    delta_table.alias('t').merge(
        df_language_bridge.alias('s'),
        ((col('s.movie_id') == col('t.movie_id'))&(col('s.id') == col('t.id')) & col('t.release_month').isin(month_list) & col('t.release_year').isin(year_list))
    ).whenMatchedUpdateAll()\
        .whenNotMatchedInsertAll()\
        .execute()

else:
    # Initial Load
    df_language_bridge.write.format('delta') \
        .mode('overwrite') \
        .partitionBy('release_year','release_month')\
        .option('path',language_bridge)\
        .save()

